### RAG pipelines  -Data Ingestion to Vector DB pipeline


In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\afiqu\Desktop\Learning And Evaulating Python\RAG with LAngChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: machine_learning.pdf
  ✓ Loaded 1 pages

Processing: python_intro.pdf
  ✓ Loaded 1 pages

Total documents loaded: 2


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-02T12:22:25+10:00', 'author': 'Md Afique Amin Zian', 'moddate': '2026-05-02T12:22:25+10:00', 'source': '..\\data\\pdf\\machine_learning.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'machine_learning.pdf', 'file_type': 'pdf'}, page_content='Machine Learning Basics \n \nMachine learning is a subset of artificial intelligence that enables systems to learn and improve \nfrom experience without being explicitly programmed. It focuses on developing computer programs \nthat can access data and use it to learn for themselves. \n \nTypes of Machine Learning: \n1. Supervised Learning: Learning with labeled data \n2. Unsupervised Learning: Finding patterns in unlabeled data \n3. Reinforcement Learning: Learning through rewards and penalties \n \nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'produ

In [4]:
### Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #show a example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f'"Content": {split_docs[0].page_content[:100]}...')  # Show first 100 characters
        print(f'"Metadata": {split_docs[0].metadata}')

    return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 2 documents into 2 chunks

Example chunk:
"Content": Machine Learning Basics 
 
Machine learning is a subset of artificial intelligence that enables syst...
"Metadata": {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-02T12:22:25+10:00', 'author': 'Md Afique Amin Zian', 'moddate': '2026-05-02T12:22:25+10:00', 'source': '..\\data\\pdf\\machine_learning.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'machine_learning.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-02T12:22:25+10:00', 'author': 'Md Afique Amin Zian', 'moddate': '2026-05-02T12:22:25+10:00', 'source': '..\\data\\pdf\\machine_learning.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'machine_learning.pdf', 'file_type': 'pdf'}, page_content='Machine Learning Basics \n \nMachine learning is a subset of artificial intelligence that enables systems to learn and improve \nfrom experience without being explicitly programmed. It focuses on developing computer programs \nthat can access data and use it to learn for themselves. \n \nTypes of Machine Learning: \n1. Supervised Learning: Learning with labeled data \n2. Unsupervised Learning: Finding patterns in unlabeled data \n3. Reinforcement Learning: Learning through rewards and penalties \n \nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'produ

### Embedding and VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [1]:
import torch
print(torch.__version__)  # Should print without errors

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Everything works!")

2.11.0+cpu


c:\Users\afiqu\Desktop\Learning And Evaulating Python\RAG with LAngChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\afiqu\Desktop\Learning And Evaulating Python\RAG with LAngChain\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\afiqu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you

✅ Everything works!


In [7]:
from sentence_transformers import SentenceTransformer
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2430.32it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\afiqu\AppData\Local\Temp\ipykernel_13940\809047281.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 2


In [9]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-02T12:22:25+10:00', 'author': 'Md Afique Amin Zian', 'moddate': '2026-05-02T12:22:25+10:00', 'source': '..\\data\\pdf\\machine_learning.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'machine_learning.pdf', 'file_type': 'pdf'}, page_content='Machine Learning Basics \n \nMachine learning is a subset of artificial intelligence that enables systems to learn and improve \nfrom experience without being explicitly programmed. It focuses on developing computer programs \nthat can access data and use it to learn for themselves. \n \nTypes of Machine Learning: \n1. Supervised Learning: Learning with labeled data \n2. Unsupervised Learning: Finding patterns in unlabeled data \n3. Reinforcement Learning: Learning through rewards and penalties \n \nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'produ

In [9]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.83it/s]

Generated embeddings with shape: (2, 384)
Adding 2 documents to vector store...
Successfully added 2 documents to vector store
Total documents in collection: 4


### Retriever Pipeline From VectorStore

In [10]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [11]:
rag_retriever.retrieve("Types of Machine Learning? and Key Features of python")

Retrieving documents for query: 'Types of Machine Learning? and Key Features of python'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.60it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_08393ad0_0',
  'content': 'Machine Learning Basics \n \nMachine learning is a subset of artificial intelligence that enables systems to learn and improve \nfrom experience without being explicitly programmed. It focuses on developing computer programs \nthat can access data and use it to learn for themselves. \n \nTypes of Machine Learning: \n1. Supervised Learning: Learning with labeled data \n2. Unsupervised Learning: Finding patterns in unlabeled data \n3. Reinforcement Learning: Learning through rewards and penalties \n \nApplications include image recognition, speech processing, and recommendation systems',
  'metadata': {'creator': 'Microsoft® Word 2016',
   'content_length': 579,
   'producer': 'Microsoft® Word 2016',
   'doc_index': 0,
   'file_type': 'pdf',
   'source_file': 'machine_learning.pdf',
   'total_pages': 1,
   'page_label': '1',
   'moddate': '2026-05-02T12:22:25+10:00',
   'page': 0,
   'creationdate': '2026-05-02T12:22:25+10:00',
   'source': '..\\da

In [27]:
rag_retriever.retrieve("What is Python in one sentence?")

Retrieving documents for query: 'What is Python in one sentence?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.90it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_9e364286_1',
  'content': 'Python Programming Introduction \n \nPython is a high-level, interpreted programming language known for its simplicity and readability. \nCreated by Guido van Rossum and first released in 1991, Python has become one of the most \npopular \nprogramming languages in the world. \n \nKey Features: \n- Easy to learn and use \n- Extensive standard library \n- Cross-platform compatibility \n- Strong community support \n \nPython is widely used in web development, data science, artificial intelligence, and automation.',
  'metadata': {'author': 'Md Afique Amin Zian',
   'creator': 'Microsoft® Word 2016',
   'page_label': '1',
   'moddate': '2026-05-02T12:22:59+10:00',
   'content_length': 502,
   'file_type': 'pdf',
   'source_file': 'python_intro.pdf',
   'creationdate': '2026-05-02T12:22:59+10:00',
   'total_pages': 1,
   'page': 0,
   'producer': 'Microsoft® Word 2016',
   'doc_index': 1,
   'source': '..\\data\\pdf\\python_intro.pdf'},
  'similarity_

### Integration VectorDB Context Pipeline with LLM output

In [14]:
### Simple RAG pipeline wuith GROQ LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### initialize the GROQ LLM
groq_api_key= os.getenv("GROQ_API_KEY")

llm=ChatGroq(api_key=groq_api_key, model_name="qwen/qwen3-32b", temperature=0.1, max_tokens=1024)

### Simple RAG function: retreive context + genrerate answer
def rag_simple(query, retreiver, llm, top_k=3):
    ### retreive the context
    results=retreiver.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else "No relevant documents found."
    if not context:
        return "No relevant documents found to answer the query."
    ### generate the answer
    prompt = f"""Use the following context to answer the question concisely.
        context:
        {context}

        Question: {query}
        Answer:""" 
    respnse = llm.invoke([prompt.format(context=context, query=query)])
    return respnse.content

In [17]:
answer=rag_simple("What is Python in one sentence?", rag_retriever, llm)    
print(answer)

Retrieving documents for query: 'What is Python in one sentence?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.50it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


<think>
Okay, the user is asking for a one-sentence definition of Python based on the provided context. Let me look at the context again.

The context starts by stating that Python is a high-level, interpreted programming language known for simplicity and readability. It mentions Guido van Rossum as the creator and the release year 1991. Also, it highlights key features like easy to learn, extensive libraries, cross-platform, and community support. It's used in web dev, data science, AI, automation.

So the answer needs to be concise. The first sentence from the context is a good start. Maybe combine the key points: high-level, interpreted, simplicity, readability, creator, release year. Also, mention its popularity and main uses. Let me check the example answer given. The example answer is: "Python is a high-level, interpreted programming language known for its simplicity and readability, created by Guido van Rossum in 1991, widely used in web development, data science, artificial int

### Enhanced RAG Pipeline Features

In [19]:
### --- Enhnaced RAG pipeline Features ----

def rag_advanced(query, retreiver, llm, top_k=5, mind_score=0.2, return_context= False):
    """RAG Pipeline with extra features
    -Return answer, sources, confidence score, and optionally the retrieved context.
    """
    results=retreiver.retrieve(query, top_k=top_k, score_threshold=mind_score)

    if not results:
        return{'answer': "No relevant documents found to answer the query.", 'sources': [], 'confidence_score': 0.0, 'context': [] if return_context else None}
    
    #prepare context for LLM
    context = "\n\n".join([doc['content'] for doc in results]) if results else "No relevant documents found."
    sources= [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page' : doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:100] + '...'  # first 200 characters as preview
    }for doc in results]

    confidence=max([doc['similarity_score'] for doc in results])

    ##generate answer
    prompt = f"""Use the following context to answer the question concisely.
        context:
        {context}

        Question: {query}
        Answer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence_score': confidence,
        'context': context if return_context else None
    }
    if return_context:
        output['context'] = context
    return output

### Example of output
results = rag_advanced("What is Python?", rag_retriever, llm, top_k=3, mind_score=0.1, return_context=True)
print("Answer:", results['answer'])
print("\nSources:")
for source in results['sources']:
    print(f" - {source['source']} (Score: {source['score']:.4f})")
    print(f"   Preview: {source['preview']}")
print("\nConfidence Score:", results['confidence_score'])
print("\nContext:\n", results['context'][:100], "...")


Retrieving documents for query: 'What is Python?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: <think>
Okay, the user is asking "What is Python?" and I need to use the provided context to answer concisely. Let me look through the context again.

The context starts by stating Python is a high-level, interpreted language known for simplicity and readability. It mentions Guido van Rossum created it in 1991. Then there are key features like easy to learn, extensive library, cross-platform, and community support. Also, it's used in web dev, data science, AI, automation.

So the answer should include that it's a high-level, interpreted language, created by Guido in 1991, popular for its simplicity. Then mention the key features briefly and the main application areas. Need to keep it concise but cover all main points. Let me structure that without being too wordy.
</think>

Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum in 1991, it is widely used in web development, data science, artificial intellig

In [20]:
### Example of output
results = rag_advanced("What is Bluetooth?", rag_retriever, llm, top_k=3, mind_score=0.1, return_context=True)
print("Answer:", results['answer'])
print("\nSources:")
for source in results['sources']:
    print(f" - {source['source']} (Score: {source['score']:.4f})")
    print(f"   Preview: {source['preview']}")
print("\nConfidence Score:", results['confidence_score'])
print("\nContext:\n", results['context'][:100], "...")

Retrieving documents for query: 'What is Bluetooth?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.73it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant documents found to answer the query.

Sources:

Confidence Score: 0.0

Context:
 [] ...
